# House Prices — Advanced Regression Techniques

Notebook for the Kaggle competition *House Prices: Advanced Regression Techniques*.

Combines techniques from the **Intermediate Machine Learning**, **Feature Engineering**, and
**Time Series** courses: missing value handling, categorical encoding, Pipelines,
cross-validation, XGBoost, mathematical feature engineering, mutual information, clustering,
PCA, target encoding, and a trend/seasonality hybrid model.

**Structure:**
1. Import and EDA
2. Advanced feature engineering
3. Mutual Information analysis
4. Preprocessing (Pipeline: imputation, one-hot encoding, target encoding, cluster and PCA features)
5. Model comparison with cross-validation
6. Tuning the best model
7. Hybrid model: market trend + seasonality + XGBoost on residuals
8. Final training and submission


## 1. Import and EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
from category_encoders import MEstimateEncoder

TRAIN_PATH = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv'
TEST_PATH = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv'

X_full = pd.read_csv(TRAIN_PATH, index_col='Id')
X_test_full = pd.read_csv(TEST_PATH, index_col='Id')

print("Train shape:", X_full.shape)
print("Test shape:", X_test_full.shape)

In [ ]:
# Remove rows without a target, separate target from predictors
X_full.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = X_full['SalePrice']
X_full = X_full.drop(['SalePrice'], axis=1)

X_full.head()

In [ ]:
# Missing values per column (only columns with at least 1 missing value)
missing = X_full.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

In [ ]:
# Target distribution
plt.figure(figsize=(8, 4))
plt.hist(y, bins=40)
plt.title('SalePrice distribution')
plt.xlabel('SalePrice')
plt.ylabel('Frequency')
plt.show()

print("SalePrice - mean:", y.mean(), "| median:", y.median())

## 2. Advanced feature engineering

I add a few hand-built features, which often help
tree-based models like XGBoost pick up on relationships that aren't obvious from the raw columns:

- **LivLotRatio**: ratio of living area to lot area
- **Spaciousness**: total floor area divided by the number of rooms
- **TotalOutsideSF**: sum of all outdoor areas (deck, porches, verandas)
- **PorchTypes**: how many different types of porch/veranda the house has
- **MedNhbdArea**: median living area in the neighborhood (*grouped transform*)

**Note on MedNhbdArea**: to avoid data leakage, the median per neighborhood is computed
**only on the training set** and then applied (mapped) to the test set — not recalculated on the
test set.

In [ ]:
def add_engineered_features(df):
    df = df.copy()
    porch_cols = ['WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch']
    for c in porch_cols:
        df[c] = df[c].fillna(0)

    df['LivLotRatio'] = df['GrLivArea'] / df['LotArea']
    df['Spaciousness'] = (df['1stFlrSF'] + df['2ndFlrSF']) / df['TotRmsAbvGrd']
    df['TotalOutsideSF'] = df[porch_cols].sum(axis=1)
    df['PorchTypes'] = (df[porch_cols] > 0).sum(axis=1)
    return df

X_full = add_engineered_features(X_full)
X_test_full = add_engineered_features(X_test_full)

nbhd_median = X_full.groupby('Neighborhood')['GrLivArea'].median()
global_median = X_full['GrLivArea'].median()

X_full['MedNhbdArea'] = X_full['Neighborhood'].map(nbhd_median)
X_test_full['MedNhbdArea'] = X_test_full['Neighborhood'].map(nbhd_median).fillna(global_median)

new_feature_cols = ['LivLotRatio', 'Spaciousness', 'TotalOutsideSF', 'PorchTypes', 'MedNhbdArea']
X_full[new_feature_cols].describe()

## 3. Mutual Information analysis

*Mutual information* measures how informative each feature
is, on its own, with respect to the target. We don't use it to drop columns (XGBoost handles
weakly informative features fine), but to understand which variables — including the ones we just
created — matter most.

In [ ]:
def make_mi_scores(X, y):
    X = X.copy()
    object_cols = X.select_dtypes(exclude=[np.number]).columns
    for colname in object_cols:
        X[colname], _ = X[colname].factorize()
    discrete_features = [pd.api.types.is_integer_dtype(t) for t in X.dtypes]
    mi_scores = mutual_info_regression(
        X.fillna(0), y, discrete_features=discrete_features, random_state=0
    )
    mi_scores = pd.Series(mi_scores, name="MI Scores", index=X.columns)
    return mi_scores.sort_values(ascending=False)

mi_scores = make_mi_scores(X_full, y)

plt.figure(figsize=(8, 6))
top = mi_scores.head(20).sort_values()
plt.barh(top.index, top.values)
plt.title("Top 20 features by Mutual Information")
plt.tight_layout()
plt.show()

mi_scores[new_feature_cols].sort_values(ascending=False)

## 4. Preprocessing

 I add three more techniques here, all wrapped as scikit-learn-compatible transformers — so they stay
**leak-safe inside cross-validation**: they get re-fit on every single training fold, exactly like
the other pipeline steps.

- **Target Encoding** (`Neighborhood`): instead of One-Hot (which for `Neighborhood`, with ~25
  categories, would create many sparse columns), we use `MEstimateEncoder`, which replaces each neighborhood with an estimate of the average
  SalePrice, smoothed toward the global mean to avoid overfitting on neighborhoods with few houses.
- **Cluster feature**: I group houses into 10
  clusters based on area/lot size, and use the assigned cluster as a new categorical feature.
- **PCA-derived features**: I reduce a group of
  correlated area-related features to their principal components, which often isolate patterns
  like "large, recently renovated house" into a single variable.

In [ ]:
class ClusterFeature(BaseEstimator, TransformerMixin):
    """Assigns a K-Means cluster (fit on train, reused on test) as a new feature."""

    def __init__(self, n_clusters=10, random_state=0):
        self.n_clusters = n_clusters
        self.random_state = random_state

    def fit(self, X, y=None):
        X = X.fillna(X.mean())
        self.mean_ = X.mean()
        self.std_ = X.std()
        X_scaled = (X - self.mean_) / self.std_
        self.kmeans_ = KMeans(
            n_clusters=self.n_clusters, n_init=10, random_state=self.random_state
        )
        self.kmeans_.fit(X_scaled)
        return self

    def transform(self, X):
        X = X.fillna(X.mean())
        X_scaled = (X - self.mean_) / self.std_
        cluster = self.kmeans_.predict(X_scaled)
        return pd.DataFrame({"Cluster": cluster}, index=X.index)


class PCAFeatures(BaseEstimator, TransformerMixin):
    """Creates principal components (fit on train, reused on test) as new features."""

    def __init__(self, n_components=3, random_state=0):
        self.n_components = n_components
        self.random_state = random_state

    def fit(self, X, y=None):
        X = X.fillna(X.mean())
        self.mean_ = X.mean()
        self.std_ = X.std()
        X_scaled = (X - self.mean_) / self.std_
        self.pca_ = PCA(n_components=self.n_components, random_state=self.random_state)
        self.pca_.fit(X_scaled)
        return self

    def transform(self, X):
        X = X.fillna(X.mean())
        X_scaled = (X - self.mean_) / self.std_
        comps = self.pca_.transform(X_scaled)
        cols = [f"PC{i+1}" for i in range(comps.shape[1])]
        return pd.DataFrame(comps, columns=cols, index=X.index)

In [ ]:
numerical_cols = X_full.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X_full.select_dtypes(exclude=[np.number]).columns if c != 'Neighborhood']

cluster_input_cols = ['LotArea', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'GrLivArea']
pca_input_cols = ['GarageArea', 'YearRemodAdd', 'TotalBsmtSF', 'GrLivArea']

print(f"Numerical columns: {len(numerical_cols)}")
print(f"Categorical columns (one-hot): {len(categorical_cols)}")
print("Neighborhood -> handled separately via target encoding")

numerical_transformer = SimpleImputer(strategy='median')

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols),
    ('nbhd_target_enc', MEstimateEncoder(m=1.0), ['Neighborhood']),
    ('cluster', ClusterFeature(n_clusters=10, random_state=0), cluster_input_cols),
    ('pca', PCAFeatures(n_components=3, random_state=0), pca_input_cols),
])

In [ ]:
# Validation split, used only for model comparison during development
X_train, X_valid, y_train, y_valid = train_test_split(
    X_full, y, train_size=0.8, test_size=0.2, random_state=0
)

## 5. Model comparison with cross-validation

I compare a few Random Forest configurations, a regularized
linear baseline (Ridge), and XGBoost — this time on the dataset
enriched with all the additional features.

I use cross-validation over the whole training set, rather
than a single split, for a more reliable estimate.

In [ ]:
def build_pipeline(model):
    return Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])

candidates = {
    "Random Forest (100 trees)": RandomForestRegressor(n_estimators=100, random_state=0),
    "Random Forest (300 trees, max_depth=15)": RandomForestRegressor(
        n_estimators=300, max_depth=15, random_state=0
    ),
    "Ridge (linear baseline)": Ridge(alpha=10.0),
    "XGBoost (default)": XGBRegressor(random_state=0),
}

cv_results = {}
for name, model in candidates.items():
    pipeline = build_pipeline(model)
    scores = -1 * cross_val_score(
        pipeline, X_full, y, cv=5, scoring='neg_mean_absolute_error'
    )
    cv_results[name] = scores.mean()
    print(f"{name}: average MAE = {scores.mean():,.0f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.barh(list(cv_results.keys()), list(cv_results.values()))
plt.xlabel('Average MAE (5-fold cross-validation)')
plt.title('Model comparison')
plt.tight_layout()
plt.show()

## 6. Tuning the best model

XGBoost is typically the best-performing model on this dataset. I try to improve it by increasing
the number of trees and lowering the learning rate.

In [ ]:
tuned_xgb = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    random_state=0,
)

tuned_pipeline = build_pipeline(tuned_xgb)
scores = -1 * cross_val_score(
    tuned_pipeline, X_full, y, cv=5, scoring='neg_mean_absolute_error'
)
print(f"XGBoost tuned: average MAE = {scores.mean():,.0f}")
cv_results["XGBoost (tuned)"] = scores.mean()

## 7. Hybrid model: market trend + seasonality + XGBoost on residuals

This section adapts the **Time Series** course to a dataset that *isn't* really a time series.

- **House Prices is cross-sectional, not sequential.** Each row is an independent house sale, not
  a point in a continuous chronological series. Techniques built around that idea — **lag/lead features**
  and **multistep forecasting** — don't have a meaningful
  equivalent here, and forcing them in would just add noise.
- **Trend and seasonality do transfer, in an adapted form.** Real estate prices genuinely move
  with macro trends (year-over-year appreciation) and with the month a sale happens (seasonal
  demand). That's exactly what the *Trend*, *Seasonality*, and *Linear Regression with Time Series*
  exercises model — just applied here to `YrSold`/`MoSold` instead of a daily sales series.
- **The Hybrid Models exercise transfers directly**, as an architecture: `Model 1` fits the
  time-based component (trend + seasonality), `Model 2` (XGBoost) fits the **residual** using the
  house's own characteristics. This cleanly separates "when it was sold" from "what the house is
  like".

In [ ]:
class HybridTrendSeasonalXGB(BaseEstimator, RegressorMixin):

    """Model 1 (LinearRegression) captures the market trend over time and the seasonality of the
    sale month. Model 2 (XGBoost) learns to predict the residual relative to that trend,
    based on the house's characteristics.
    """

    def __init__(self, preprocessor, xgb_params=None, trend_order=2, fourier_order=2, base_year=2006):
        self.preprocessor = preprocessor
        self.xgb_params = xgb_params or {}
        self.trend_order = trend_order
        self.fourier_order = fourier_order
        self.base_year = base_year

    def _time_features(self, X):
        # Manual trend + Fourier seasonality terms (equivalent to DeterministicProcess +
        # CalendarFourier from the course), built so they generalize to any YrSold/MoSold
        # without needing an in-sample/out-of-sample index like statsmodels' DeterministicProcess.
        t = (X['YrSold'] - self.base_year) * 12 + (X['MoSold'] - 1)
        feats = {}
        for o in range(1, self.trend_order + 1):
            feats[f'trend_t{o}'] = t ** o
        for k in range(1, self.fourier_order + 1):
            feats[f'fourier_sin_{k}'] = np.sin(2 * np.pi * k * X['MoSold'] / 12)
            feats[f'fourier_cos_{k}'] = np.cos(2 * np.pi * k * X['MoSold'] / 12)
        return pd.DataFrame(feats, index=X.index)

    def fit(self, X, y):
        X_time = self._time_features(X)
        self.model_1_ = LinearRegression()
        self.model_1_.fit(X_time, y)
        y_fit = self.model_1_.predict(X_time)
        y_resid = y - y_fit

        self.model_2_ = Pipeline(steps=[
            ('preprocessor', clone(self.preprocessor)),
            ('model', XGBRegressor(**self.xgb_params)),
        ])
        self.model_2_.fit(X, y_resid)
        return self

    def predict(self, X):
        X_time = self._time_features(X)
        market_pred = self.model_1_.predict(X_time)
        resid_pred = self.model_2_.predict(X)
        return market_pred + resid_pred


hybrid_model = HybridTrendSeasonalXGB(
    preprocessor=preprocessor,
    xgb_params=dict(n_estimators=1000, learning_rate=0.05, max_depth=4, random_state=0),
    trend_order=2,
    fourier_order=2,
    base_year=int(X_full['YrSold'].min()),
)

scores = -1 * cross_val_score(
    hybrid_model, X_full, y, cv=5, scoring='neg_mean_absolute_error'
)
print(f"Hybrid model (trend + seasonality + XGBoost): average MAE = {scores.mean():,.0f}")
cv_results["Hybrid (trend + seasonality + XGBoost)"] = scores.mean()

In [ ]:
best_model_name = min(cv_results, key=cv_results.get)
print("Best model:", best_model_name, "| MAE:", round(cv_results[best_model_name]))

plt.figure(figsize=(8, 5))
plt.barh(list(cv_results.keys()), list(cv_results.values()))
plt.xlabel('Average MAE (5-fold cross-validation)')
plt.title('Final model comparison (including the hybrid model)')
plt.tight_layout()
plt.show()

## 8. Final training and submission

I pick whichever model scored best in cross-validation above — plain tuned XGBoost or the hybrid
model — retrain it on the **entire** available training set (not just the validation split), then
generate predictions on the test set and the `submission.csv` file in the format required by the
competition.

In [ ]:
if best_model_name == "Hybrid (trend + seasonality + XGBoost)":
    final_model = HybridTrendSeasonalXGB(
        preprocessor=preprocessor,
        xgb_params=dict(n_estimators=1000, learning_rate=0.05, max_depth=4, random_state=0),
        trend_order=2,
        fourier_order=2,
        base_year=int(X_full['YrSold'].min()),
    )
else:
    final_model = build_pipeline(tuned_xgb)

final_model.fit(X_full, y)
test_preds = final_model.predict(X_test_full)

output = pd.DataFrame({
    'Id': X_test_full.index,
    'SalePrice': test_preds
})
output.to_csv('submission.csv', index=False)

output.head()

## Conclusions

These are the actual results from running this notebook on the real competition data.

**Feature Engineering:**

Of the five hand-built features, `MedNhbdArea` (median living area per neighborhood) came out by
far the most informative (MI score 0.47), ahead of `Spaciousness` (0.25) and `TotalOutsideSF`
(0.22), with `PorchTypes` (0.11) and `LivLotRatio` (0.09) contributing less on their own. This
confirms the intuition behind that feature: where a house is matters roughly as much as its raw
size, and a neighborhood-level price signal is worth encoding explicitly rather than leaving the
model to infer it indirectly from the (target-encoded) `Neighborhood` column alone.

**Model comparison:**

With default hyperparameters, Random Forest (100 trees: MAE 17,585; 300 trees: 17,608), Ridge
(18,589) and XGBoost (17,516) all landed within about 1,000 MAE of each other. Ridge being the
weakest, but not by a huge margin, suggests that after target encoding, one-hot encoding, and the
engineered features, a large part of the signal is already close to linear-the tree-based models'
advantage comes from the remaining non-linear interactions, not from the preprocessing itself.

**Tuning:**

Tuning XGBoost (more trees, lower learning rate) brought the cross-validation MAE from 17,516 down
to 15,585 — roughly an 11% improvement over the default configuration.

**The hybrid model did not beat tuned XGBoost:**

The trend + seasonality + XGBoost hybrid scored 15,847 MAE, slightly *worse* than plain tuned
XGBoost (15,585). This is a meaningful negative result, not a failure of the approach: it indicates
that whatever timing effect exists in `YrSold`/`MoSold` is weak enough, and already well captured
by XGBoost treating them as ordinary features, that isolating it into a separate linear stage adds
no extra lift here.

**Leaderboard score: 0.13268 (RMSLE).**